# RAG Assignment: Baseline vs. Re-Ranking

**Two-stage retrieval over a single complex PDF, comparing a dense Bi-Encoder baseline against a Cross-Encoder re-ranker.**

- **Dataset:** *CATAN* base-game *Game Rules & Almanac* (official rulebook PDF, 16 pages). A rulebook is a deliberately good stress document: it is dense with **exceptions** ("except when…"), **negations** ("you may not…"), and **corner cases** whose wording overlaps heavily with unrelated rules — exactly where a lexical/dense baseline stumbles and a cross-encoder helps.
- **Bi-Encoder (retriever):** `sentence-transformers/all-MiniLM-L6-v2`
- **Cross-Encoder (re-ranker):** `cross-encoder/ms-marco-MiniLM-L-6-v2`
- **LLM (answer generation):** Gemini API (`gemini-flash-lite-latest`). **Add your key to a `.env` file** (`GEMINI_API_KEY=...`); everything except the final LLM answer runs with no key.

**Pipelines compared**

| | Pipeline 1 — Baseline | Pipeline 2 — Advanced Re-Ranking |
|---|---|---|
| Retrieve | Bi-Encoder → **Top 10** by cosine similarity | Same **Top 10** candidates |
| Select | **Top 3** directly | Cross-Encoder re-scores all 10 → **Top 3** |
| Generate | Top 3 → LLM → **Answer 1** | Re-ranked Top 3 → LLM (identical prompt) → **Answer 2** |

## 0. Setup & Environment

Run order: top to bottom. First run downloads the two models (~160 MB) and caches them.
To generate `Answer 1` / `Answer 2`, add your key to a `.env` file next to this notebook
(`GEMINI_API_KEY=your_key_here`); retrieval, re-ranking, and latency profiling run without one.

In [1]:
# If a package is missing in your conda env, uncomment and run once:
# %pip install -q sentence-transformers pymupdf numpy pandas tqdm google-genai python-dotenv

import os, time, re, textwrap
import numpy as np
import pandas as pd
import pymupdf  # PyMuPDF
from sentence_transformers import SentenceTransformer, CrossEncoder
from dotenv import load_dotenv

pd.set_option("display.max_colwidth", 90)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# ---- Dataset ----
PDF_PATH = "catan_rulebook.pdf"      # the rulebook (same folder as this notebook)

# ---- Models ----
BI_ENCODER_NAME    = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# ---- Retrieval settings ----
K_RETRIEVE = 10   # Top-K pulled from the vector store (both pipelines)
K_FINAL    = 3    # Top-N handed to the LLM

# ---- Chunking settings ----
CHUNK_TARGET_CHARS = 700   # approx chunk size
CHUNK_OVERLAP_SENTS = 2    # sentences of overlap between consecutive chunks

# ---- LLM settings (Gemini) ----
# Put your key in a .env file next to this notebook:  GEMINI_API_KEY=your_key_here
# (.env is git-ignored, so the key is never pushed to the public repo.)
load_dotenv(dotenv_path=".env")  # loads GEMINI_API_KEY from .env
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "PASTE_YOUR_GEMINI_API_KEY_HERE")
GEMINI_MODEL   = "gemini-flash-lite-latest"

# device: Apple Silicon (mps) -> cuda -> cpu
import torch
if torch.backends.mps.is_available():
    DEVICE = "mps"
elif torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print("Device:", DEVICE)

/opt/homebrew/Caskroom/miniconda/base/envs/global/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1019 19:31:24.568000 78570 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Device: mps


## 1. Load the Document

Extract text page-by-page with PyMuPDF. We strip the dotted "Almanac Index" lines (pure
table-of-contents noise like `Longest Road . . . . . 9`) so chunks carry real rules text.

In [2]:
def clean_page_text(t: str) -> str:
    t = t.replace("­", "")                      # soft hyphens
    lines = []
    for ln in t.split("\n"):
        # drop dotted index/TOC lines
        if re.search(r"(\.\s){6,}", ln) or ln.count(".") > 12:
            continue
        lines.append(ln)
    return re.sub(r"\s+", " ", " ".join(lines)).strip()

def load_pdf_pages(path: str):
    doc = pymupdf.open(path)
    pages = []
    for i in range(doc.page_count):
        txt = clean_page_text(doc[i].get_text())
        if len(txt) > 80:                            # skip near-empty / cover pages
            pages.append({"page": i, "text": txt})
    return pages

pages = load_pdf_pages(PDF_PATH)
print(f"Loaded {len(pages)} content pages from '{PDF_PATH}'")
print("\n--- sample (page", pages[3]["page"], ") ---")
print(textwrap.fill(pages[3]["text"][:600], 100))

Loaded 15 content pages from 'catan_rulebook.pdf'

--- sample (page 4 ) ---
5 b) Settlement Y Requires: Brick, Lumber, Wool, & Grain Take special note of the “Distance Rule” Y:
you may only build a settlement at an intersection if all 3 of the adjacent intersections are vacant
(i.e., none are occupied by any settlements or cities—even yours). Each of your settlements must
connect to at least 1 of your own roads. Regardless of whose turn it is (i.e., during any production
phase), when a terrain hex produces resources, you receive 1 resource card for each settlement you
have adjacent to that terrain hex. Each settlement is worth 1 victory point. c) City Y Requires: 3
Or


## 2. Chunking

Sentence-aware chunking: pack whole sentences up to ~`CHUNK_TARGET_CHARS`, with a small
sentence overlap so a rule split across a boundary is not lost. Both pipelines index the
**exact same** chunk corpus.

In [3]:
def split_sentences(t: str):
    return [p.strip() for p in re.split(r"(?<=[.!?])\s+", t) if p.strip()]

def chunk_pages(pages, target_chars=CHUNK_TARGET_CHARS, overlap_sents=CHUNK_OVERLAP_SENTS):
    chunks = []
    for pg in pages:
        buf, cur_len = [], 0
        for s in split_sentences(pg["text"]):
            buf.append(s); cur_len += len(s) + 1
            if cur_len >= target_chars:
                chunks.append({"id": len(chunks), "page": pg["page"], "text": " ".join(buf)})
                buf = buf[-overlap_sents:]
                cur_len = sum(len(x) + 1 for x in buf)
        if buf and (not chunks or chunks[-1]["text"] != " ".join(buf)):
            chunks.append({"id": len(chunks), "page": pg["page"], "text": " ".join(buf)})
    return chunks

chunks = chunk_pages(pages)
CHUNK_TEXTS = [c["text"] for c in chunks]
wlens = [len(c["text"].split()) for c in chunks]
print(f"Built {len(chunks)} chunks | words per chunk: min {min(wlens)}, mean {np.mean(wlens):.0f}, max {max(wlens)}")
print("\n--- example chunk", chunks[34]["id"], "(page", chunks[34]["page"], ") ---")
print(textwrap.fill(chunks[34]["text"], 100))

Built 99 chunks | words per chunk: min 21, mean 131, max 235

--- example chunk 34 (page 6 ) ---
You cannot trade or give away development cards. You may only play 1 development card during your
turn— either 1 knight card or 1 progress card. You can play the card at any time, even before you
roll the dice. You may not, however, play a card that you bought during the same turn. Exception: If
you buy a card and it is a victory point card Y that brings you to 10 points, you may immediately
reveal this card (and all other VP cards) and win the game. You only reveal victory point cards when
the game is over—once you or an opponent reaches 10+ victory points and declares victory. Distance
Rule You may only build a settlement on an unoccupied intersection Y and only if none of the 3
adjacent intersections contains a settlement or city.


## 3. Pipeline 1 — Index with the Bi-Encoder (the Vector Store)

The Bi-Encoder embeds each chunk **independently** into a fixed vector. We L2-normalize, so
an inner product equals cosine similarity. The matrix of normalized embeddings **is** our
vector store; a query is embedded the same way and scored by cosine against every chunk.

In [4]:
print("Loading Bi-Encoder:", BI_ENCODER_NAME)
bi_encoder = SentenceTransformer(BI_ENCODER_NAME, device=DEVICE)

t0 = time.perf_counter()
chunk_embeddings = bi_encoder.encode(
    CHUNK_TEXTS, normalize_embeddings=True, convert_to_numpy=True, batch_size=64, show_progress_bar=False
)
index_build_time = time.perf_counter() - t0
print(f"Embedded {chunk_embeddings.shape[0]} chunks -> dim {chunk_embeddings.shape[1]} "
      f"(one-time index build: {index_build_time*1000:.0f} ms)")

class VectorStore:
    """Tiny in-memory cosine-similarity store (normalized embeddings -> inner product)."""
    def __init__(self, embeddings, chunks):
        self.emb = embeddings
        self.chunks = chunks
    def search(self, query_vec, k=K_RETRIEVE):
        sims = self.emb @ query_vec              # cosine (vectors are normalized)
        order = np.argsort(-sims)[:k]
        return [(int(i), float(sims[i])) for i in order]

store = VectorStore(chunk_embeddings, chunks)

def bi_encode_query(query: str) -> np.ndarray:
    return bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True)[0]

Loading Bi-Encoder: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9889.51it/s]

Embedded 99 chunks -> dim 384 (one-time index build: 650 ms)


## 4. Pipeline 2 — The Cross-Encoder Re-Ranker

A Cross-Encoder takes the **(query, chunk) pair together** and runs joint self-attention
across all tokens of both, producing a single relevance score. It is far more accurate than
cosine over independent embeddings — and far more expensive — so we only apply it to the 10
candidates the baseline already retrieved.

In [5]:
print("Loading Cross-Encoder:", CROSS_ENCODER_NAME)
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME, device=DEVICE)

def cross_encoder_rerank(query, candidate_ids):
    pairs = [[query, chunks[i]["text"]] for i in candidate_ids]
    scores = cross_encoder.predict(pairs)
    ranked = sorted(zip(candidate_ids, scores), key=lambda x: -x[1])
    return [(int(i), float(s)) for i, s in ranked]

Loading Cross-Encoder: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 8593.88it/s]

## 5. LLM Wrapper (Gemini, key loaded from `.env`)

`call_llm` reads `GEMINI_API_KEY` from the `.env` file (loaded in section 0) and calls Gemini,
with retry on transient 5xx and a graceful message if the free-tier quota is hit. With no key it
returns a clearly labelled sentinel so the whole notebook still runs — retrieval, re-ranking,
rank-shift and latency analysis need no key. **Both pipelines use the identical prompt
template**, so any difference in the answer comes only from *which chunks* were supplied.

In [6]:
_PLACEHOLDERS = {"", "PASTE_YOUR_GEMINI_API_KEY_HERE"}
LLM_NOT_CONFIGURED = "[LLM NOT CONFIGURED — add GEMINI_API_KEY to the .env file (see section 0) to generate answers]"

PROMPT_TEMPLATE = """You are a precise assistant answering questions about a board game rulebook.
Answer the QUESTION using ONLY the information in the CONTEXT passages below.
If the context does not contain the answer, say "The provided context does not contain the answer."
Be concise and quote the specific rule, including any exceptions or restrictions.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

def build_prompt(question, selected_ids):
    ctx = "\n\n".join(f"[Passage {n+1} | page {chunks[i]['page']}]\n{chunks[i]['text']}"
                      for n, i in enumerate(selected_ids))
    return PROMPT_TEMPLATE.format(context=ctx, question=question)

def llm_is_configured():
    return bool(GEMINI_API_KEY) and not str(GEMINI_API_KEY).startswith("PASTE_YOUR_")

def call_llm(prompt: str) -> str:
    if not llm_is_configured():
        return LLM_NOT_CONFIGURED
    from google import genai
    from google.genai import errors as genai_errors
    client = genai.Client(api_key=GEMINI_API_KEY)
    import re as _re
    last_err = None
    for attempt in range(6):
        try:
            resp = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
            return (resp.text or "").strip()
        except genai_errors.ServerError as e:      # transient 5xx (e.g. 503 high demand)
            last_err = e
            time.sleep(2 ** attempt)
        except genai_errors.ClientError as e:      # 429 RESOURCE_EXHAUSTED
            if getattr(e, "code", None) != 429 and "RESOURCE_EXHAUSTED" not in str(e):
                raise
            m = _re.search(r"retryDelay'?: ?'?(\d+)s", str(e))
            delay = int(m.group(1)) if m else 20
            if delay <= 90 and attempt < 5:        # short delay => per-minute limit: wait and retry
                time.sleep(delay + 2)
                continue
            return "[RATE LIMITED — Gemini quota reached; re-run later or use a billed key]"
    raise last_err

print("LLM configured:", llm_is_configured(), "| model:", GEMINI_MODEL)

LLM configured: True | model: gemini-flash-lite-latest


## 6. The Two Pipelines + a Unified `run_query`

`run_query` executes both pipelines over the same candidate set and records: the Top-10 ids,
baseline Top-3, re-ranked Top-3, the per-chunk rank shift, both answers, and the timings used
for latency profiling.

In [7]:
def run_query(question, k_retrieve=K_RETRIEVE, k_final=K_FINAL, generate=True, verbose=False):
    # ---------- Pipeline 1: Baseline retrieval (Bi-Encoder) ----------
    t0 = time.perf_counter()
    qvec = bi_encode_query(question)
    hits = store.search(qvec, k=k_retrieve)              # [(id, cosine), ...] sorted desc
    t_baseline_retrieval = time.perf_counter() - t0      # embed query + vector search

    cand_ids       = [i for i, _ in hits]
    baseline_scores = {i: s for i, s in hits}
    baseline_top    = cand_ids[:k_final]

    # ---------- Pipeline 2: Cross-Encoder re-ranking ----------
    t0 = time.perf_counter()
    reranked = cross_encoder_rerank(question, cand_ids)  # [(id, ce_score), ...] sorted desc
    t_rerank = time.perf_counter() - t0                  # extra cost on top of baseline

    reranked_ids    = [i for i, _ in reranked]
    rerank_scores   = {i: s for i, s in reranked}
    reranked_top    = reranked_ids[:k_final]

    # ---------- rank-shift bookkeeping ----------
    base_rank = {cid: r + 1 for r, cid in enumerate(cand_ids)}
    rr_rank   = {cid: r + 1 for r, cid in enumerate(reranked_ids)}

    # ---------- LLM answers (identical prompt template) ----------
    answer1 = answer2 = None
    t_llm1 = t_llm2 = 0.0
    if generate:
        p1 = build_prompt(question, baseline_top)
        p2 = build_prompt(question, reranked_top)
        s = time.perf_counter(); answer1 = call_llm(p1); t_llm1 = time.perf_counter() - s
        s = time.perf_counter(); answer2 = call_llm(p2); t_llm2 = time.perf_counter() - s

    result = {
        "question": question,
        "candidate_ids": cand_ids,
        "baseline_top": baseline_top,
        "reranked_top": reranked_top,
        "baseline_scores": baseline_scores,
        "rerank_scores": rerank_scores,
        "base_rank": base_rank,
        "rr_rank": rr_rank,
        "answer1": answer1,
        "answer2": answer2,
        "t_baseline_retrieval": t_baseline_retrieval,
        "t_rerank": t_rerank,
        "t_llm1": t_llm1,
        "t_llm2": t_llm2,
    }
    if verbose:
        pretty_print_result(result)
    return result

def _chunk_preview(i, width=95):
    return textwrap.shorten(chunks[i]["text"], width)

def pretty_print_result(r):
    print("Q:", r["question"])
    print("\nPipeline 1 — Baseline Top-3:")
    for i in r["baseline_top"]:
        print(f"  #{r['base_rank'][i]} [chunk {i}, p{chunks[i]['page']}] cos={r['baseline_scores'][i]:.3f}  {_chunk_preview(i)}")
    print("\nPipeline 2 — Re-ranked Top-3:")
    for i in r["reranked_top"]:
        moved = r["base_rank"][i]
        tag = f"(was baseline #{moved})"
        print(f"  #{r['rr_rank'][i]} [chunk {i}, p{chunks[i]['page']}] ce={r['rerank_scores'][i]:.2f} {tag}  {_chunk_preview(i)}")
    if r["answer1"] is not None:
        print("\n--- ANSWER 1 (baseline) ---\n", textwrap.fill(r["answer1"], 100))
        print("\n--- ANSWER 2 (re-ranked) ---\n", textwrap.fill(r["answer2"], 100))

## 7. Sanity Checks on Straightforward Queries

On easy, well-phrased factual queries the baseline already retrieves the right chunk, so the two
pipelines mostly agree — re-ranking matters on the hard cases in the next section. Below we run 8
sanity queries and report how much their baseline Top-3 and re-ranked Top-3 overlap; then we show
the full Answer 1 vs Answer 2 for one of them.

In [8]:
SANITY_QUERIES = [
    "How many victory points is a city worth?",
    "What resources are required to build a settlement?",
    "What does it cost to upgrade a settlement to a city?",
    "How many roads, settlements, and cities does each player start with?",
    "How many victory points is the Longest Road card worth?",
    "How many resource cards does a city produce compared to a settlement?",
    "How many victory points do you need to win the game?",
    "What is the Distance Rule for building settlements?",
]

print("Top-3 overlap between baseline and re-ranked pipelines (higher = they agree):\n")
for q in SANITY_QUERIES:
    r = run_query(q, generate=False)
    overlap = len(set(r["baseline_top"]) & set(r["reranked_top"]))
    print(f"  [{overlap}/3 overlap]  baseline {r['baseline_top']} | re-ranked {r['reranked_top']}  <-  {q}")

print("\n" + "=" * 100)
print("Full Answer 1 vs Answer 2 for one sanity query (both pipelines should agree):")
_ = run_query(SANITY_QUERIES[0], verbose=True)

Top-3 overlap between baseline and re-ranked pipelines (higher = they agree):

  [2/3 overlap]  baseline [30, 26, 83] | re-ranked [30, 96, 83]  <-  How many victory points is a city worth?


  [1/3 overlap]  baseline [18, 77, 78] | re-ranked [18, 94, 19]  <-  What resources are required to build a settlement?
  [3/3 overlap]  baseline [30, 29, 19] | re-ranked [29, 30, 19]  <-  What does it cost to upgrade a settlement to a city?
  [1/3 overlap]  baseline [6, 68, 70] | re-ranked [69, 6, 11]  <-  How many roads, settlements, and cities does each player start with?
  [1/3 overlap]  baseline [47, 39, 26] | re-ranked [47, 38, 10]  <-  How many victory points is the Longest Road card worth?


  [2/3 overlap]  baseline [19, 29, 30] | re-ranked [30, 19, 58]  <-  How many resource cards does a city produce compared to a settlement?
  [2/3 overlap]  baseline [26, 38, 83] | re-ranked [83, 24, 38]  <-  How many victory points do you need to win the game?
  [2/3 overlap]  baseline [67, 68, 18] | re-ranked [67, 18, 70]  <-  What is the Distance Rule for building settlements?

Full Answer 1 vs Answer 2 for one sanity query (both pipelines should agree):


Q: How many victory points is a city worth?

Pipeline 1 — Baseline Top-3:
  #1 [chunk 30, p5] cos=0.673  Each city is worth 2 victory points. You receive double resource production (2 resource [...]
  #2 [chunk 26, p4] cos=0.599  © 2015 Catan GmbH 1 Victory Point! Reveal this card on your turn if, with it, you reach [...]
  #3 [chunk 83, p13] cos=0.586  Therefore, you only need 8 more victory points to win the game! Hint: In Catan, you will [...]

Pipeline 2 — Re-ranked Top-3:
  #1 [chunk 30, p5] ce=6.94 (was baseline #1)  Each city is worth 2 victory points. You receive double resource production (2 resource [...]
  #2 [chunk 96, p15] ce=6.29 (was baseline #9)  You begin the game with 2 settlements and 2 roads. Each settlement is worth 1 victory [...]
  #3 [chunk 83, p13] ce=2.34 (was baseline #3)  Therefore, you only need 8 more victory points to win the game! Hint: In Catan, you will [...]

--- ANSWER 1 (baseline) ---
 Based on the provided context, "Each city is worth 2 victory poi

## 8. Failure Mode Analysis (Stress Testing)

Eleven hand-crafted queries where the Bi-Encoder baseline degrades (the chunk that actually
answers the question is NOT in its Top-3) but the Cross-Encoder restores relevance (it promotes
that chunk into the Top-3). They span the four failure families named in the assignment:

1. **Keyword Trap / Distractor** — a passage with high lexical overlap to the query is irrelevant,
   while the true answer uses different terms (e.g. the *number tokens* "…there is no 7…" chunk vs.
   the robber rule).
2. **Negation & Constraint** — the governing rule is a prohibition ("you may **not** …"), whose
   embedding sits near the positive passages that describe when the action *is* allowed.
3. **Buried Vital Chunk** — the answer is a single clause buried under many superficially similar
   passages (e.g. robber *movement* passages hide where the robber *starts*).
4. **Exception / Corner Case** — an easily-missed "except when…" / empty-supply / tie clause.

In [9]:
FAILURE_QUERIES = [
    {"category": "Keyword Trap / Distractor",
     "query": "What happens when a 7 is rolled?"},
    {"category": "Negation & Constraint",
     "query": "Can I trade with the bank during another player's turn?"},
    {"category": "Negation & Constraint",
     "query": "Can I give my resource cards to another player as a gift?"},
    {"category": "Negation & Constraint",
     "query": "Can development cards be returned to the supply pile?"},
    {"category": "Negation & Constraint",
     "query": "Can two players trade the same resource type, like two wool for one wool?"},
    {"category": "Buried Vital Chunk",
     "query": "Where does the robber start the game?"},
    {"category": "Buried Vital Chunk",
     "query": "When are victory point cards revealed?"},
    {"category": "Buried Vital Chunk",
     "query": "Can I play a knight card before rolling the dice?"},
    {"category": "Exception / Corner Case",
     "query": "Can I move the robber onto the desert hex?"},
    {"category": "Exception / Corner Case",
     "query": "Can I buy a development card when the deck is empty?"},
    {"category": "Exception / Corner Case",
     "query": "What happens if the supply runs out of a resource when several players are owed it?"},
]
print(f"{len(FAILURE_QUERIES)} failure queries across "
      f"{len(set(f['category'] for f in FAILURE_QUERIES))} categories")

11 failure queries across 4 categories


In [10]:
failure_results = []
for fq in FAILURE_QUERIES:
    print("=" * 100)
    print(f"[{fq['category']}]")
    r = run_query(fq["query"], generate=False, verbose=True)
    r["category"] = fq["category"]
    failure_results.append(r)
    print()

[Keyword Trap / Distractor]
Q: What happens when a 7 is rolled?

Pipeline 1 — Baseline Top-3:
  #1 [chunk 54, p9] cos=0.396  1010 N Number Tokens The 18 number tokens are marked with the numerals “2” through “12.” [...]
  #2 [chunk 58, p9] cos=0.351  You receive your income for each terrain hex adjacent to your settlements or cities every [...]
  #3 [chunk 70, p11] cos=0.335  The player who rolls highest is the starting player and begins. The starting player [...]

Pipeline 2 — Re-ranked Top-3:
  #1 [chunk 64, p10] ce=2.42 (was baseline #4)  “A” and “B” do not receive a brick resource card. This lasts until the robber is moved [...]
  #2 [chunk 65, p10] ce=1.96 (was baseline #6)  If you hold an odd number of cards, round down (e.g., if you have 9 resource cards, you [...]
  #3 [chunk 54, p9] ce=0.52 (was baseline #1)  1010 N Number Tokens The 18 number tokens are marked with the numerals “2” through “12.” [...]

[Negation & Constraint]


Q: Can I trade with the bank during another player's turn?

Pipeline 1 — Baseline Top-3:
  #1 [chunk 36, p6] cos=0.476  Coleman cannot build on the intersections marked “B.” He can only build at intersection [...]
  #2 [chunk 15, p3] cos=0.421  Important: Players may only trade with the player whose turn it is. The other players may [...]
  #3 [chunk 12, p3] cos=0.406  Turn Overview Unless you’re using the Starting Set-Up for Experienced Players, the oldest [...]

Pipeline 2 — Re-ranked Top-3:
  #1 [chunk 80, p13] ce=8.77 (was baseline #9)  1414 Trade After you roll for resource production, you may trade with other players [...]
  #2 [chunk 15, p3] ce=7.81 (was baseline #2)  Important: Players may only trade with the player whose turn it is. The other players may [...]
  #3 [chunk 14, p3] ce=4.81 (was baseline #4)  You receive 2 resource cards for each city you own that borders that hex. If there are [...]

[Negation & Constraint]
Q: Can I give my resource cards to another player as a 

Q: When are victory point cards revealed?

Pipeline 1 — Baseline Top-3:
  #1 [chunk 26, p4] cos=0.683  © 2015 Catan GmbH 1 Victory Point! Reveal this card on your turn if, with it, you reach [...]
  #2 [chunk 23, p4] cos=0.660  The first player to have 3 knight cards in front of themself receives the special card [...]
  #3 [chunk 81, p13] cos=0.657  • You may not trade like resources (e.g., 2 wool for 1 wool). V Victory Point Cards [...]

Pipeline 2 — Re-ranked Top-3:
  #1 [chunk 81, p13] ce=5.85 (was baseline #3)  • You may not trade like resources (e.g., 2 wool for 1 wool). V Victory Point Cards [...]
  #2 [chunk 35, p6] ce=5.52 (was baseline #5)  You only reveal victory point cards when the game is over—once you or an opponent reaches [...]
  #3 [chunk 24, p4] ce=5.42 (was baseline #6)  You may only reveal them during your turn and when you are sure that you have 10 victory [...]

[Buried Vital Chunk]
Q: Can I play a knight card before rolling the dice?

Pipeline 1 — Baseline Top-3

## 9. Rank-Shift Analysis

For every failure query, the table shows each Top-10 candidate's **baseline rank → re-ranked
rank**. `moved_into_top3 = True` marks a chunk the Cross-Encoder promoted from rank 4–10 into
the final Top-3 (the ones the LLM now actually sees).

In [11]:
rows = []
for r in failure_results:
    for cid in r["candidate_ids"]:
        br, rr = r["base_rank"][cid], r["rr_rank"][cid]
        rows.append({
            "query": r["question"][:45],
            "category": r["category"],
            "chunk": cid,
            "page": chunks[cid]["page"],
            "baseline_rank": br,
            "rerank_rank": rr,
            "delta": br - rr,                       # +ve = promoted
            "in_base_top3": br <= K_FINAL,
            "in_rerank_top3": rr <= K_FINAL,
            "moved_into_top3": (br > K_FINAL) and (rr <= K_FINAL),
            "preview": _chunk_preview(cid, 60),
        })
rank_shift_df = pd.DataFrame(rows)

# headline: chunks promoted INTO the top-3 by re-ranking
promoted = rank_shift_df[rank_shift_df["moved_into_top3"]].copy()
print("Chunks PROMOTED into the Top-3 by the Cross-Encoder:\n")
display(promoted[["query", "category", "chunk", "page", "baseline_rank", "rerank_rank", "preview"]]
        .reset_index(drop=True))

Chunks PROMOTED into the Top-3 by the Cross-Encoder:



,query,category,chunk,page,baseline_rank,rerank_rank,preview
0,What happens when a 7 is rolled?,Keyword Trap / Distractor,64,10,4,1,“A” and “B” do not receive a brick resource card. This [...]
1,What happens when a 7 is rolled?,Keyword Trap / Distractor,65,10,6,2,"If you hold an odd number of cards, round down (e.g., [...]"
2,Can I trade with the bank during another play,Negation & Constraint,14,3,4,3,You receive 2 resource cards for each city you own [...]
3,Can I trade with the bank during another play,Negation & Constraint,80,13,9,1,"1414 Trade After you roll for resource production, you [...]"
4,Can I give my resource cards to another playe,Negation & Constraint,80,13,7,2,"1414 Trade After you roll for resource production, you [...]"
5,Can development cards be returned to the supp,Negation & Constraint,29,5,4,2,New rule variant: see Combined Trade/Build Phase Y. [...]
6,Can development cards be returned to the supp,Negation & Constraint,19,4,7,3,c) City Y Requires: 3 Ore & 2 Grain You may only [...]
7,"Can two players trade the same resource type,",Negation & Constraint,51,8,5,3,If you have built a settlement or city at a harbor Y [...]
8,"Can two players trade the same resource type,",Negation & Constraint,81,13,8,2,"• You may not trade like resources (e.g., 2 wool for 1 [...]"
9,Where does the robber start the game?,Buried Vital Chunk,69,11,4,2,Set-Up Phase Begin the “set-up phase” after you build [...]


In [12]:
# full per-query rank-shift view (sorted by re-ranked rank)
for r in failure_results:
    sub = (rank_shift_df[rank_shift_df["query"] == r["question"][:45]]
           .sort_values("rerank_rank"))
    print(f"\n### {r['category']}: {r['question']}")
    display(sub[["chunk", "page", "baseline_rank", "rerank_rank", "delta",
                 "in_base_top3", "in_rerank_top3", "preview"]].reset_index(drop=True))


### Keyword Trap / Distractor: What happens when a 7 is rolled?


,chunk,page,baseline_rank,rerank_rank,delta,in_base_top3,in_rerank_top3,preview
0,64,10,4,1,3,False,True,“A” and “B” do not receive a brick resource card. This [...]
1,65,10,6,2,4,False,True,"If you hold an odd number of cards, round down (e.g., [...]"
2,54,9,1,3,-2,True,True,1010 N Number Tokens The 18 number tokens are marked [...]
3,95,15,5,4,1,False,False,The numbers on the round tokens are depicted in [...]
4,70,11,3,5,-2,True,False,The player who rolls highest is the starting player [...]
5,67,10,10,6,4,False,False,You share in all of the resource production of each [...]
6,58,9,2,7,-5,True,False,You receive your income for each terrain hex adjacent [...]
7,98,15,7,8,-1,False,False,"If, for example, a “10” is rolled, all terrain hexes [...]"
8,9,2,8,9,-1,False,False,"Next, read the Game Rules and start to play. And [...]"
9,33,6,9,10,-1,False,False,7 D Desert The desert is the only terrain hex that [...]



### Negation & Constraint: Can I trade with the bank during another player's turn?


,chunk,page,baseline_rank,rerank_rank,delta,in_base_top3,in_rerank_top3,preview
0,80,13,9,1,8,False,True,"1414 Trade After you roll for resource production, you [...]"
1,15,3,2,2,0,True,True,Important: Players may only trade with the player [...]
2,14,3,4,3,1,False,True,You receive 2 resource cards for each city you own [...]
3,36,6,1,4,-3,True,False,Coleman cannot build on the intersections marked “B.” [...]
4,62,10,6,5,1,False,False,"1111 Resource Trade In the second phase of your turn, [...]"
5,34,6,5,6,-1,False,False,You cannot trade or give away development cards. You [...]
6,12,3,3,7,-4,True,False,Turn Overview Unless you’re using the Starting Set-Up [...]
7,79,12,8,8,0,False,False,"Even if it is not your turn, you should offer trades [...]"
8,49,8,7,9,-2,False,False,Special Case: If your longest road is broken and you [...]
9,20,4,10,10,0,False,False,"Each has a different effect (see 4.b, below). [...]"



### Negation & Constraint: Can I give my resource cards to another player as a gift?


...

,chunk,page,baseline_rank,rerank_rank,delta,in_base_top3,in_rerank_top3,preview
0,21,4,2,1,1,True,True,Proceed as follows: (1) You must move the robber Y [...]
1,43,7,4,2,2,False,True,Place the knight card face up in front of you. You [...]
2,20,4,3,3,0,True,True,"Each has a different effect (see 4.b, below). [...]"
3,65,10,6,4,2,False,False,"If you hold an odd number of cards, round down (e.g., [...]"
4,63,10,1,5,-4,True,False,"Liam, the white player, would like to build a road. He [...]"
5,42,7,7,6,1,False,False,H Harbors Harbors allow you to trade resources more [...]
6,66,10,8,7,1,False,False,"(3) After discarding occurs, you also steal 1 resource [...]"
7,33,6,5,8,-3,False,False,7 D Desert The desert is the only terrain hex that [...]
8,67,10,10,9,1,False,False,You share in all of the resource production of each [...]
9,95,15,9,10,-1,False,False,The numbers on the round tokens are depicted in [...]



### Exception / Corner Case: Can I buy a development card when the deck is empty?


,chunk,page,baseline_rank,rerank_rank,delta,in_base_top3,in_rerank_top3,preview
0,19,4,5,1,4,False,True,c) City Y Requires: 3 Ore & 2 Grain You may only [...]
1,34,6,1,2,-1,True,True,You cannot trade or give away development cards. You [...]
2,33,6,2,3,-1,True,True,7 D Desert The desert is the only terrain hex that [...]
3,81,13,6,4,2,False,False,"• You may not trade like resources (e.g., 2 wool for 1 [...]"
4,28,5,4,5,-1,False,False,Return the resource cards to the supply stacks. You [...]
5,56,9,7,6,1,False,False,There are 2 each of 3 varieties: Road Building: If you [...]
6,57,9,8,7,1,False,False,You may play only 1 development card Y during your [...]
7,60,9,3,8,-5,True,False,If there are not enough resource cards to give every [...]
8,59,9,9,9,0,False,False,"Example: See Illustration J. Loren, the blue player, [...]"
9,91,14,10,10,0,False,False,"However, if you find any components missing or [...]"



### Exception / Corner Case: What happens if the supply runs out of a resource when several players are owed it?


,chunk,page,baseline_rank,rerank_rank,delta,in_base_top3,in_rerank_top3,preview
0,60,9,1,1,0,True,True,If there are not enough resource cards to give every [...]
1,59,9,2,2,0,True,True,"Example: See Illustration J. Loren, the blue player, [...]"
2,14,3,4,3,1,False,True,You receive 2 resource cards for each city you own [...]
3,28,5,3,4,-1,True,False,Return the resource cards to the supply stacks. You [...]
4,64,10,5,5,0,False,False,“A” and “B” do not receive a brick resource card. This [...]
5,13,3,6,6,0,False,False,"Tip: For advanced players, we recommend combining the [...]"
6,56,9,10,7,3,False,False,There are 2 each of 3 varieties: Road Building: If you [...]
7,65,10,9,8,1,False,False,"If you hold an odd number of cards, round down (e.g., [...]"
8,61,9,8,9,-1,False,False,They can be 2 of the same resource or 2 different [...]
9,27,5,7,10,-3,False,False,"6 Almanac This “Catan Almanac” contains detailed, [...]"


## 10. Latency Profiling

Baseline retrieval = embed the query + cosine search over the vector store.
Re-ranking adds the Cross-Encoder scoring of all 10 (query, chunk) pairs **on top of** the
baseline. We average over repeats; LLM generation time is excluded here (network-bound) and
reported separately if a key is set.

In [13]:
REPEATS = 5
lat_rows = []
for fq in FAILURE_QUERIES:
    q = fq["query"]
    bt, rt = [], []
    for _ in range(REPEATS):
        r = run_query(q, generate=False)
        bt.append(r["t_baseline_retrieval"] * 1000)   # ms
        rt.append(r["t_rerank"] * 1000)               # ms
    base_ms = float(np.mean(bt))
    rr_ms   = float(np.mean(rt))
    lat_rows.append({
        "query": q[:50],
        "pipeline1_baseline_ms": round(base_ms, 2),
        "rerank_extra_ms": round(rr_ms, 2),
        "pipeline2_total_ms": round(base_ms + rr_ms, 2),
        "slowdown_x": round((base_ms + rr_ms) / base_ms, 1),
    })

latency_df = pd.DataFrame(lat_rows)
print(f"Averaged over {REPEATS} repeats on device '{DEVICE}'. One-time index build: {index_build_time*1000:.0f} ms\n")
display(latency_df)

agg = {
    "mean Pipeline 1 baseline (ms)": latency_df["pipeline1_baseline_ms"].mean(),
    "mean re-rank extra (ms)":        latency_df["rerank_extra_ms"].mean(),
    "mean Pipeline 2 total (ms)":     latency_df["pipeline2_total_ms"].mean(),
    "mean slowdown (x)":              latency_df["slowdown_x"].mean(),
}
print("\nAggregate:")
for k, v in agg.items():
    print(f"  {k:32} {v:8.2f}")

Averaged over 5 repeats on device 'mps'. One-time index build: 650 ms



,query,pipeline1_baseline_ms,rerank_extra_ms,pipeline2_total_ms,slowdown_x
0,What happens when a 7 is rolled?,5.05,38.33,43.37,8.6
1,Can I trade with the bank during another player's,4.89,46.25,51.14,10.5
2,Can I give my resource cards to another player as,4.86,31.01,35.87,7.4
3,Can development cards be returned to the supply pi,4.74,30.11,34.85,7.4
4,"Can two players trade the same resource type, like",4.48,31.02,35.50,7.9
5,Where does the robber start the game?,4.50,30.65,35.15,7.8
6,When are victory point cards revealed?,4.66,29.88,34.54,7.4
7,Can I play a knight card before rolling the dice?,4.87,30.39,35.26,7.2
8,Can I move the robber onto the desert hex?,4.76,31.81,36.57,7.7
9,Can I buy a development card when the deck is empt,4.82,31.10,35.92,7.5



Aggregate:
  mean Pipeline 1 baseline (ms)        4.74
  mean re-rank extra (ms)             32.95
  mean Pipeline 2 total (ms)          37.68
  mean slowdown (x)                    7.95


## 11. Answer 1 vs. Answer 2 (requires `GEMINI_API_KEY` in `.env`)

With the key loaded from `.env`, this regenerates both answers for each failure query so you can
see whether the baseline (Answer 1) hallucinated, omitted a clause, or misstated a rule
because the critical chunk sat outside its Top-3, and whether re-ranking (Answer 2) fixes it.

**Note on quota:** generating answers for all 11 failure queries is 22 LLM calls. A free-tier
Gemini key caps daily requests, so some answers may come back as `[RATE LIMITED ...]`; if so,
re-run this cell later (the quota resets daily) or use a billed key. The retrieval, rank-shift,
and latency analysis above need no API and are complete for all 11 queries regardless.

In [14]:
if llm_is_configured():
    for r in failure_results:
        fresh = run_query(r["question"], generate=True)
        print("=" * 100)
        print(f"[{r['category']}]  Q: {r['question']}")
        print("\n--- ANSWER 1 (baseline Top-3) ---")
        print(textwrap.fill(fresh["answer1"], 100))
        print("\n--- ANSWER 2 (re-ranked Top-3) ---")
        print(textwrap.fill(fresh["answer2"], 100))
        print()
else:
    print(LLM_NOT_CONFIGURED)
    print("\nShowing the two prompts for the first failure query so you can inspect what each")
    print("pipeline would send to the LLM (note the different CONTEXT passages):\n")
    r = failure_results[0]
    print("########## PIPELINE 1 PROMPT (baseline Top-3) ##########\n")
    print(build_prompt(r["question"], r["baseline_top"]))
    print("\n\n########## PIPELINE 2 PROMPT (re-ranked Top-3) ##########\n")
    print(build_prompt(r["question"], r["reranked_top"]))

[Keyword Trap / Distractor]  Q: What happens when a 7 is rolled?

--- ANSWER 1 (baseline Top-3) ---
The provided context does not contain the answer.

--- ANSWER 2 (re-ranked Top-3) ---
Based on the provided context, when a "7" is rolled for resource production, none of the players
receive resources. Instead:  1. **Discarding cards:** "Each player counts their resource cards. Any
player with more than 7 resource cards (i.e., 8 or more) must choose and discard half of them.
Return discards to the supply stacks. If you hold an odd number of cards, round down (e.g., if you
have 9 resource cards, you discard 4)." 2. **Moving the robber:** "Then you (the player who rolled
the “7”) must move the robber Y to the number token Y of any other terrain hex (or to the desert Y
hex). This blocks the resource production of this hex, until the robber moves to another number
token or the desert." 3. **Stealing a card:** "After discarding occurs, you also steal 1 resource
card at random from a player wh

[Negation & Constraint]  Q: Can I trade with the bank during another player's turn?

--- ANSWER 1 (baseline Top-3) ---
The provided context does not contain the answer.

--- ANSWER 2 (re-ranked Top-3) ---
Based on the provided context: "You may not trade with the bank during another player’s turn."



[Negation & Constraint]  Q: Can I give my resource cards to another player as a gift?

--- ANSWER 1 (baseline Top-3) ---
The provided context does not contain the answer.

--- ANSWER 2 (re-ranked Top-3) ---
Based on the provided context, no. The rule states: "You may not give away cards."



[Negation & Constraint]  Q: Can development cards be returned to the supply pile?

--- ANSWER 1 (baseline Top-3) ---
The provided context does not contain the answer.

--- ANSWER 2 (re-ranked Top-3) ---
Based on the provided context, development cards never return to the supply.   Quote: "Development
cards never go back into the supply, and you cannot buy development cards if the supply is empty."
(Passage 3)



[Negation & Constraint]  Q: Can two players trade the same resource type, like two wool for one wool?

--- ANSWER 1 (baseline Top-3) ---
Based on the provided context, no, you cannot trade matching resources.   Passage 2 states:
"However, you cannot give away cards, or trade matching resources (“trade” 3 ore for 1 ore, for
example)."

--- ANSWER 2 (re-ranked Top-3) ---
The provided context states: "You may not trade like resources (e.g., 2 wool for 1 wool)."



...

[Buried Vital Chunk]  Q: When are victory point cards revealed?

--- ANSWER 1 (baseline Top-3) ---
Based on the provided context, victory point cards are revealed under the following conditions and
restrictions:  * "You must keep victory point cards hidden. You may only reveal them during your
turn and when you are sure that you have 10 victory points—that is, to win the game." (Passage 2,
similar details in Passage 1 and Passage 3). * "Of course, you can reveal them after the end of the
game if someone else wins." (Passage 2) / "You only reveal your victory point cards when you or
someone else wins the game!" (Passage 3).

--- ANSWER 2 (re-ranked Top-3) ---
Based on the provided context, victory point cards are revealed: - "when you or someone else wins
the game!" / "when the game is over—once you or an opponent reaches 10+ victory points and declares
victory." - "during your turn and when you are sure that you have 10 victory points—that is, to win
the game." - "if someone else wins"

[Buried Vital Chunk]  Q: Can I play a knight card before rolling the dice?

--- ANSWER 1 (baseline Top-3) ---
The provided context does not contain the answer.

--- ANSWER 2 (re-ranked Top-3) ---
Yes. According to Passage 1, "You can play the card at any time, even before you roll the dice."



[Exception / Corner Case]  Q: Can I move the robber onto the desert hex?

--- ANSWER 1 (baseline Top-3) ---
Yes. According to the rules, "(1) You must move the robber Y immediately to the number token of any
other terrain hex or to the desert Y hex."

--- ANSWER 2 (re-ranked Top-3) ---
Yes. According to the rules, "(1) You must move the robber Y immediately to the number token of any
other terrain hex or to the desert Y hex." (Passage 1; see also Passage 2 and Passage 3).



[Exception / Corner Case]  Q: Can I buy a development card when the deck is empty?

--- ANSWER 1 (baseline Top-3) ---
The provided context does not contain the answer.

--- ANSWER 2 (re-ranked Top-3) ---
Based on the provided context, the answer is no.   Rule: "you cannot buy development cards if the
supply is empty." (Passage 1)



[Exception / Corner Case]  Q: What happens if the supply runs out of a resource when several players are owed it?

--- ANSWER 1 (baseline Top-3) ---
Based on the provided context: "If there are not enough resource cards to give every player all the
production they earn, then no player receives any of that resource that turn."

--- ANSWER 2 (re-ranked Top-3) ---
Based on the provided context, "If there are not enough resource cards to give every player all the
production they earn, then no player receives any of that resource that turn."



## 12. Analysis & Report

The written report (see **`report.md`** in this repo) covers the four required areas:

1. **Architecture Comparison** — Bi-Encoder (independent embeddings, cosine) vs. Cross-Encoder
   (joint self-attention over the (query, chunk) pair).
2. **Qualitative & Rank-Shift Analysis** — the promotions in section 9 (e.g. the robber "7"
   rule jumping baseline **#4 → #1**, the "trade with the bank" prohibition **#9 → #1**) and
   the Answer 1 vs. Answer 2 differences from section 11.
3. **Latency Profiling** — the table in section 10.
4. **Scaling Question** — why we cannot run the Cross-Encoder over the whole corpus from the
   start (it must score every (query, chunk) pair at query time and cannot be pre-computed,
   unlike the Bi-Encoder index).

> **Note on authenticity:** the assignment requires the report to be in *your own words* and
> not AI-generated. `report.md` is a factual draft with the measured numbers filled in —
> rewrite the prose in your own words before submitting.